# DFI GI checkpoint review (Gemini + cached report MD)

Runs every checkpoint in `data/GI/GI_DFI.json` against one `*_llm.md` report.

## Setup
```bash
cd /path/to/test_GI_reports
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python -m ipykernel install --user --name=gi-reports --display-name="Python (gi-reports)"
# .env must contain: GEMINI_API_KEY=... and optional GEMINI_MODEL=gemini-3.1-flash-lite
jupyter notebook notebooks/gi_checkpoint_review.ipynb
```

**In Jupyter:** select kernel **Python (gi-reports)** (top right).

## Debug outputs
- What is stored in Gemini **context cache** (report MD)
- What is sent per **checkpoint batch** (user prompt only when cache works)
- **Token usage** per call + totals
- **Flags** (violations)

Set `CHECKPOINT_LIMIT = None` to run all checkpoints (101). Uses parallel API calls via `MAX_WORKERS`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from gi_review import (
    GEMINI_31_FLASH_LITE_PRICING,
    build_cached_report_content,
    build_checkpoint_prompt,
    create_report_cache,
    estimate_cost_usd,
    estimate_tokens,
    format_cost_summary,
    load_checkpoints,
    load_env,
    load_report_md,
    run_all_checkpoints,
)
from google import genai

REPORT_MD = PROJECT_ROOT / "data/Reports_md/1945958_1_9557514024035__Q2611274887_3__B48DCB2E653347EF98F17B8E5B54A5C3_llm.md"
CHECKPOINTS_JSON = PROJECT_ROOT / "data/GI/GI_DFI.json"

# None = all checkpoints. Use a small number while debugging.
CHECKPOINT_LIMIT = None
MAX_WORKERS = 12

# Requires PROJECT_ROOT/.env with GEMINI_API_KEY and optional GEMINI_MODEL
api_key, model = load_env(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Model:", model)
print("Report:", REPORT_MD.name)
print("Checkpoints file:", CHECKPOINTS_JSON.name)

Project root: /Users/williambrun/Documents/test_GI_reports
Model: gemini-3.1-flash-lite
Report: 1945958_1_9557514024035__Q2611274887_3__B48DCB2E653347EF98F17B8E5B54A5C3_llm.md
Checkpoints file: GI_DFI.json


/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other 

In [ ]:
report_md = load_report_md(REPORT_MD)
checkpoints = load_checkpoints(CHECKPOINTS_JSON)
cached_payload = build_cached_report_content(report_md)

print("Report chars:", len(report_md))
print("Report est. tokens:", estimate_tokens(report_md))
print("Cached payload chars:", len(cached_payload))
print("Cached payload est. tokens:", estimate_tokens(cached_payload))
print("Total checkpoints in JSON:", len(checkpoints))
print("\n--- CACHED CONTENT PREVIEW (first 1200 chars) ---\n")
print(cached_payload[:1200])
print("\n... [truncated] ...")

Report chars: 11618
Report est. tokens: 2904
Cached payload chars: 11766
Cached payload est. tokens: 2941
Total checkpoints in JSON: 101

--- CACHED CONTENT PREVIEW (first 1200 chars) ---

You are reviewing a QIMAone DFI H&B PSI inspection report.
Use ONLY the markdown report below as evidence. Do not invent facts.

<report>
# Inspection Report — PASS

> LLM-optimized export from QIMAone JSON. Critical findings are at the top and bottom; routine PASS checks are compacted.

## Executive Summary

| Field | Value |
| --- | --- |
| Overall result | PASS |
| Inspection ID | 1945958 |
| Report ID | 1888799 |
| Inspection date | 2026-05-26 |
| Inspection type | Pre-Shipment Inspection |
| Workflow | DFI (H&B) - PSI STANDARD - HL |
| Product | guardian OMEGA-3 FISH OIL 1000mg SOFTGEL 90s |
| SKU | 9557514024035 |
| Tests result | PASS |
| Workmanship result | PASS |
| Picked cartons | 8 |
| Total cartons | 27 |

## Attention Items (read first)

- Important remark: 1. 100% of the total ordered 

In [ ]:
client = genai.Client(api_key=api_key)

cache_name, cache_usage, cache_error = create_report_cache(client, model, report_md)

print("Cache created:", bool(cache_name))
print("Cache name:", cache_name or "(none)")
print("Cache create usage:", cache_usage)
if not cache_name:
    print("Cache error:", cache_error)
    print("Fallback: full report will be sent with each checkpoint prompt.")

Cache created: True
Cache name: cachedContents/4rra2lnwnhggigz37p9ywy6yh0gvqvnyoiie19kn
Cache create usage: UsageStats(prompt_tokens=0, cached_tokens=0, output_tokens=0, total_tokens=3764)


In [ ]:
sample = checkpoints[0]
sample_prompt = build_checkpoint_prompt(sample)

print("--- PER-CHECKPOINT BATCH PREVIEW (checkpoint 1) ---\n")
print(sample_prompt)
print("\nBatch est. tokens:", estimate_tokens(sample_prompt))
print("md_sections:", sample.get("md_sections"))

--- PER-CHECKPOINT BATCH PREVIEW (checkpoint 1) ---

Checkpoint ID: preliminary.product_matches_booking
Section: before_starting
Requirement: Product inspected must match what was booked.
Look in these report sections: Executive Summary, Products & Quantities, Additional Fields
Flag if any of:
- Product in report does not match booked product
- SKU or reference order mismatch with booking

Search the cached inspection report.
Does the report violate this checkpoint?
Return ONLY valid JSON (no markdown fences):
{"violates": true|false, "reason": "short explanation", "evidence": "quote or reference from report"}
If insufficient evidence in the report text, set violates=false and explain in reason.

Batch est. tokens: 162
md_sections: ['Executive Summary', 'Products & Quantities', 'Additional Fields']


In [ ]:
import time

started = time.time()
run = run_all_checkpoints(
    REPORT_MD,
    CHECKPOINTS_JSON,
    project_root=PROJECT_ROOT,
    limit=CHECKPOINT_LIMIT,
    max_workers=MAX_WORKERS,
)
elapsed = time.time() - started

summary = format_cost_summary(run)
summary["elapsed_seconds"] = round(elapsed, 1)
print(json.dumps({k: v for k, v in summary.items() if k != "per_checkpoint"}, indent=2))

{
  "model": "gemini-3.1-flash-lite",
  "report_path": "/Users/williambrun/Documents/test_GI_reports/data/Reports_md/1945958_1_9557514024035__Q2611274887_3__B48DCB2E653347EF98F17B8E5B54A5C3_llm.md",
  "report_chars": 11618,
  "report_est_tokens": 2904,
  "cache_created": true,
  "cache_name": "cachedContents/cwh6ufd9hhybx4hjldufh5yr1k4mah1cyfz5tkdw",
  "cache_create_tokens": 3764,
  "checkpoints_run": 5,
  "flags": 0,
  "totals": {
    "prompt_tokens": 19685,
    "cached_tokens": 18820,
    "output_tokens": 452,
    "total_tokens": 23901
  }
}


In [ ]:
flags = run.flagged()
print(f"FLAGS: {len(flags)} / {len(run.checkpoint_results)} checkpoints\n")

for item in flags:
    print("=" * 72)
    print(item.checkpoint_id)
    print("Section:", item.section)
    print("Reason:", item.reason)
    print("Evidence:", item.evidence)

if not flags:
    print("No violations flagged in this run.")

# Also available in summary JSON
print(f"\n(summary flags_count: {summary.get('flags_count', len(flags))})")

FLAGS: 0 / 5 checkpoints

No violations flagged in this run.


In [ ]:
usage_rows = []
for r in run.checkpoint_results:
    usage_rows.append(
        {
            "checkpoint_id": r.checkpoint_id,
            "section": r.section,
            "violates": r.violates,
            "prompt_tokens": r.usage.prompt_tokens,
            "cached_tokens": r.usage.cached_tokens,
            "output_tokens": r.usage.output_tokens,
            "total_tokens": r.usage.total_tokens,
            "error": r.error,
        }
    )

headers = [
    "checkpoint_id",
    "violates",
    "prompt_tokens",
    "cached_tokens",
    "output_tokens",
    "total_tokens",
    "error",
]
print("\t".join(headers))
for row in usage_rows:
    print("\t".join(str(row[h]) for h in headers))

checkpoint_id	violates	prompt_tokens	cached_tokens	output_tokens	total_tokens	error
preliminary.product_matches_booking	False	3903	3764	145	4048	
preliminary.qimaone_platform	False	3896	3764	83	3979	
cover.supplier_factory_name	False	3979	3764	62	4041	
cover.product_name_not_sku	False	3989	3764	95	4084	
cover.cover_photo	False	3918	3764	67	3985	


In [ ]:
totals = run.total_usage
cost = summary["cost_usd"]["total"]
print("TOKEN TOTALS")
print(f"  prompt (all input):     {totals.prompt_tokens:,}")
print(f"  cached (from cache):    {totals.cached_tokens:,}")
print(f"  new input (billable):   {int(cost['new_input_tokens']):,}")
print(f"  output:                 {totals.output_tokens:,}")
print(f"  total billed tokens:    {totals.total_tokens:,}")
print()
print("COST (Gemini 3.1 Flash-Lite — June 2026 rates)")
print(f"  input  ${GEMINI_31_FLASH_LITE_PRICING['input_per_m']}/M:  ${cost['input_cost_usd']:.4f}")
print(f"  cached ${GEMINI_31_FLASH_LITE_PRICING['cached_input_per_m']}/M: ${cost['cached_input_cost_usd']:.4f}")
print(f"  output ${GEMINI_31_FLASH_LITE_PRICING['output_per_m']}/M: ${cost['output_cost_usd']:.4f}")
print(f"  cache create:           ${cost['cache_create_cost_usd']:.4f}")
print(f"  TOTAL:                  ${cost['total_cost_usd']:.4f}")
print()
print("CACHE vs BATCH")
print(f"  report est. tokens (cached once): {run.report_est_tokens:,}")
print(f"  cache create tokens:              {run.cache_usage.total_tokens:,}")
print(f"  checkpoints run:                  {len(run.checkpoint_results)}")
print(f"  elapsed seconds:                  {summary.get('elapsed_seconds', 'n/a')}")
if usage_rows:
    n = len(usage_rows)
    avg_prompt = sum(r["prompt_tokens"] for r in usage_rows) / n
    avg_cached = sum(r["cached_tokens"] for r in usage_rows) / n
    print(f"  avg prompt tokens / checkpoint:   {avg_prompt:.1f}")
    print(f"  avg cached tokens / checkpoint:   {avg_cached:.1f}")

TOKEN TOTALS
  prompt (new per call):  19,685
  cached (from cache):    18,820
  output:                 452
  total billed tokens:    23,901

CACHE vs BATCH
  report est. tokens (cached once): 2,904
  cache create tokens:              3,764
  checkpoints run:                  5
  avg new prompt tokens / checkpoint: 3937.0
  avg cached tokens / checkpoint:     3764.0


In [ ]:
# Inspect one full API exchange (first flagged, else first checkpoint)
inspect = flags[0] if flags else run.checkpoint_results[0]

print("CHECKPOINT:", inspect.checkpoint_id)
print("\n--- USER PROMPT SENT ---\n")
print(inspect.user_prompt)
print("\n--- MODEL RAW RESPONSE ---\n")
print(inspect.raw_response)
if inspect.error:
    print("\nERROR:", inspect.error)

CHECKPOINT: preliminary.product_matches_booking

--- USER PROMPT SENT ---

Checkpoint ID: preliminary.product_matches_booking
Section: before_starting
Requirement: Product inspected must match what was booked.
Look in these report sections: Executive Summary, Products & Quantities, Additional Fields
Flag if any of:
- Product in report does not match booked product
- SKU or reference order mismatch with booking

Search the cached inspection report.
Does the report violate this checkpoint?
Return ONLY valid JSON (no markdown fences):
{"violates": true|false, "reason": "short explanation", "evidence": "quote or reference from report"}
If insufficient evidence in the report text, set violates=false and explain in reason.

--- MODEL RAW RESPONSE ---

{
  "violates": false,
  "reason": "The product name and SKU in the report match the booking details provided in the Executive Summary and Products & Quantities sections.",
  "evidence": "Executive Summary: Product = guardian OMEGA-3 FISH OIL 1